In [ ]:

import pickle
from pathlib import Path

import numpy as np
import pandas as pd

# Paths and settings
BASE = Path("")
SOURCE_DIR = BASE / ""
EMBEDDING_DIR = BASE / ""
OUTPUT_DIR = BASE / ""

MODALITIES = {
    "email": "email_labeled.csv",
    "http": "http_labeled.csv",
    "file": "file_labeled.csv",
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Daily mean pooling for each user and modality
for modality, source_filename in MODALITIES.items():
    source_path = SOURCE_DIR / source_filename
    embedding_input_dir = EMBEDDING_DIR / modality
    modality_output_dir = OUTPUT_DIR / modality

    modality_output_dir.mkdir(parents=True, exist_ok=True)

    if not source_path.exists():
        raise FileNotFoundError(f"File not found: {source_path}")

    if not embedding_input_dir.exists():
        raise FileNotFoundError(f"Embedding directory not found: {embedding_input_dir}")

    # Preserve the same event order used during embedding generation
    events = pd.read_csv(source_path, usecols=["user", "date"], low_memory=False)
    events["date"] = pd.to_datetime(events["date"], errors="coerce")

    if events["user"].isna().any() or events["date"].isna().any():
        raise ValueError(f"[{modality}] Missing user or invalid date values")

    events = events.sort_values(["user", "date"], kind="stable").reset_index(drop=True)

    for user, user_events in events.groupby("user", sort=False):
        embedding_path = embedding_input_dir / f"{modality}_{user}_embedding.pkl"

        if not embedding_path.exists():
            raise FileNotFoundError(f"Embedding file not found: {embedding_path}")

        with open(embedding_path, "rb") as f:
            vectors = pickle.load(f)

        if not isinstance(vectors, np.ndarray) or vectors.ndim != 2:
            raise ValueError(f"[{modality}/{user}] Invalid embedding array")

        if vectors.shape[0] != len(user_events):
            raise ValueError(
                f"[{modality}/{user}] Row mismatch: "
                f"events={len(user_events)}, embeddings={vectors.shape[0]}"
            )

        if not np.isfinite(vectors).all():
            raise ValueError(f"[{modality}/{user}] Embeddings contain NaN or Inf")

        # Aggregate event embeddings by day
        day_codes, unique_days = pd.factorize(
            user_events["date"].dt.normalize(),
            sort=False
        )

        sums = np.zeros((len(unique_days), vectors.shape[1]), dtype=np.float32)
        np.add.at(sums, day_codes, vectors)

        counts = np.bincount(day_codes).astype(np.float32)
        mean_vectors = sums / counts[:, None]

        output = {
            "user": user,
            "modality": modality,
            "dates": pd.to_datetime(unique_days).strftime("%Y-%m-%d").tolist(),
            "embeddings": mean_vectors.astype(np.float32),
        }

        output_path = modality_output_dir / f"{modality}_{user}_meanpooled.pkl"

        with open(output_path, "wb") as f:
            pickle.dump(output, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"[{modality.upper()}] Mean pooled embeddings saved to {modality_output_dir}")

print(f"Mean pooling completed: {OUTPUT_DIR}")
